In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
print("alok")

alok


In [2]:
df=pd.read_csv("../Data/Cleaned/hr_cleaned.csv")
df_original=df.copy()
df.head(100000)

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,AnnualIncome,Attritionsign,overtimedone,age_group
0,50,No,Travel_Rarely,277,Sales,9,4,Marketing,1,1,...,1,2,1,0,0,0,232440,0,1,46-55
1,28,No,Travel_Rarely,1379,Research & Development,13,2,Other,1,2,...,3,2,4,3,0,3,46332,0,1,26-35
2,22,Yes,Travel_Frequently,1263,Research & Development,3,4,Life Sciences,1,3,...,5,3,0,0,0,0,35100,1,1,18-25
3,41,No,Travel_Rarely,447,Research & Development,5,3,Life Sciences,1,4,...,3,1,3,2,1,2,82380,0,0,36-45
4,36,No,Travel_Rarely,759,Research & Development,29,4,Life Sciences,1,5,...,3,2,10,9,7,8,40896,0,0,36-45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,31,No,Travel_Rarely,205,Sales,7,2,Technical Degree,0,99996,...,3,2,9,7,7,8,81516,0,0,26-35
99996,53,No,Non-Travel,660,Research & Development,0,5,Medical,1,99997,...,2,2,4,2,0,2,157032,0,1,46-55
99997,55,Yes,Travel_Rarely,718,Research & Development,2,3,Medical,1,99998,...,2,3,5,2,0,5,238104,1,1,46-55
99998,40,No,Travel_Rarely,1306,Research & Development,25,2,Technical Degree,0,99999,...,2,4,8,8,4,7,39276,0,0,36-45


In [3]:
#converting attrition
df['Attrition'] = df['Attrition'].map({'Yes':1,'No':0})

In [4]:
#encode text columns and creating x and y
le = LabelEncoder()

for col in df.select_dtypes(include='object'):
    df[col] = le.fit_transform(df[col].astype(str))

In [5]:
X = df.drop(['Attrition', 'Attritionsign'],axis=1)
y = df['Attrition']

In [6]:
#model training
X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.2,random_state=42)

In [7]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=50,
    max_depth=5,
    min_samples_leaf=20,
    min_samples_split=10,
    random_state=42
)
model.fit(X_train,y_train)

,n_estimators,50
,criterion,'gini'
,max_depth,5
,min_samples_split,10
,min_samples_leaf,20
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [8]:
test_predictions = model.predict(X_test)
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test,test_predictions)*100

print("model Accuracy =", round(accuracy,2))

model Accuracy = 88.16


In [9]:
#probability of leaving and result table
all_probabilities = model.predict_proba(X)
dashboard_df=df_original.copy()
dashboard_df['Leave_Probability'] = all_probabilities[:, 1]

dashboard_df['Leave_Percentage'] = (
    dashboard_df['Leave_Probability'] * 100).round(2)
print(dashboard_df['Leave_Percentage'])

0        12.98
1        24.00
2        50.26
3        10.02
4         6.79
         ...  
99995     9.45
99996    13.11
99997    14.20
99998     7.71
99999     4.66
Name: Leave_Percentage, Length: 100000, dtype: float64


In [10]:
def risk_category(value):
    if value >= 70:
        return "High Risk"
    elif value >= 40:
        return "Medium Risk"
    else:
        return "Low Risk"

dashboard_df['Risk_Category']=dashboard_df['Leave_Percentage'].apply(risk_category)

In [11]:
dashboard_df['Risk_Category'].value_counts()

Risk_Category
Low Risk       92210
Medium Risk     6879
High Risk        911
Name: count, dtype: int64

In [12]:
dashboard_df['Model_Accuracy'] = round(accuracy, 2)

In [13]:
dashboard_df[['EmployeeNumber','Age','Department','JobRole','OverTime',
        'MonthlyIncome','Leave_Probability','Leave_Percentage','Risk_Category','Model_Accuracy']].head()

,EmployeeNumber,Age,Department,JobRole,OverTime,MonthlyIncome,Leave_Probability,Leave_Percentage,Risk_Category,Model_Accuracy
0,1,50,Sales,Manager,Yes,19370,0.129811,12.98,Low Risk,88.16
1,2,28,Research & Development,Laboratory Technician,Yes,3861,0.239967,24.00,Low Risk,88.16
2,3,22,Research & Development,Research Scientist,Yes,2925,0.502576,50.26,Medium Risk,88.16
3,4,41,Research & Development,Healthcare Representative,No,6865,0.100184,10.02,Low Risk,88.16
4,5,36,Research & Development,Laboratory Technician,No,3408,0.067873,6.79,Low Risk,88.16


In [14]:
print(dashboard_df.shape)
print(dashboard_df.columns.tolist())

(100000, 43)
['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeCount', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'Over18', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'AnnualIncome', 'Attritionsign', 'overtimedone', 'age_group', 'Leave_Probability', 'Leave_Percentage', 'Risk_Category', 'Model_Accuracy']


In [31]:
# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
# import matplotlib.pyplot as plt

# accuracy = accuracy_score(y_test,test_predictions)
# precision = precision_score(y_test,test_predictions)
# recall = recall_score(y_test,test_predictions)
# f1 = f1_score(y_test,test_predictions)

# metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
# scores = [accuracy, precision, recall, f1]

# plt.figure(figsize=(8,5))
# plt.bar(metrics, scores)
# plt.ylim(0,1)
# plt.title('Model Performance Comparison')
# plt.ylabel('Score')
# plt.show()
# print("Accuracy:", round(accuracy*100,2))
# print("Precision:", round(precision*100,2))
# print("Recall:", round(recall*100,2))
# print("F1 Score:", round(f1*100,2))

In [16]:
dashboard_df.to_csv("../Data/Processed/new_ml_dashboard_data_100k.csv",index=False)